<a href="https://colab.research.google.com/github/Gauravd1710/legal-doc-analyzer/blob/main/notebooks/05_ledgar_addition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
import numpy as np

BASE      = '/content/drive/MyDrive/LegalDocAnalyzer'
REPO_PATH = '/content/legal-doc-analyzer'

sys.path.append(f'{BASE}/src')

# reload all configs
with open(f'{BASE}/src/label_config.json') as f:
    label_config = json.load(f)

with open(f'{BASE}/src/entity_map.json') as f:
    ENTITY_MAP = json.load(f)

LABEL_LIST = label_config['label_list']
LABEL2ID   = label_config['label2id']
ID2LABEL   = {int(k): v for k, v in label_config['id2label'].items()}

print("Config loaded")
print(f"   Labels     : {LABEL_LIST}")
print(f"   Num labels : {label_config['num_labels']}")

In [ ]:
!pip install spacy datasets transformers -q
!python -m spacy download en_core_web_trf -q

import spacy
import re
from datasets import load_dataset, load_from_disk
from collections import Counter

print(" Libraries loaded")
print(f"   spaCy version : {spacy.__version__}")

# load the best spaCy transformer model
print("\nLoading spaCy en_core_web_trf...")
nlp = spacy.load("en_core_web_trf")

print(" spaCy model loaded")
print(f"   Pipeline : {nlp.pipe_names}")

In [ ]:
print("Loading LEDGAR dataset from HuggingFace...")

ledgar = load_dataset(
    "lex_glue",
    "ledgar",
    verification_mode='no_checks'
)

#  train split
ledgar_train = ledgar['train']

print(f" LEDGAR loaded")
print(f"   Total clauses : {len(ledgar_train)}")
print(f"   Features      : {ledgar_train.features}")

#  examples
print(f"\n=== SAMPLE CLAUSES ===")
for i in range(3):
    text  = ledgar_train[i]['text']
    label = ledgar_train[i]['label']
    print(f"\nClause {i+1} (label={label}):")
    print(f"  {text[:200]}...")

iob2 converter

In [ ]:
SPACY_TO_ENTITY = {
    "PERSON" : "PARTY",
    "ORG"    : "PARTY",
    "DATE"   : "DATE",
    "TIME"   : "DATE",
    "MONEY"  : "AMOUNT",
    "PERCENT": "AMOUNT",
    "GPE"    : "JURISDICTION",
    "LOC"    : "JURISDICTION",
    "LAW"    : "TERM",
}

def clause_to_iob2(text, nlp_model):
    # run spaCy on the clause
    doc = nlp_model(text)

    if not doc or len(doc) == 0:
        return None, None

    tokens   = []
    ner_tags = []

    for token in doc:
        # skip whitespace tokens
        if token.is_space:
            continue

        word = token.text
        tokens.append(word)


        ent_type = token.ent_type_
        ent_iob  = token.ent_iob_

        if ent_iob == "O" or ent_type not in SPACY_TO_ENTITY:

            ner_tags.append(LABEL2ID["O"])
        else:

            entity_label = SPACY_TO_ENTITY[ent_type]

            if ent_iob == "B":
                ner_tags.append(LABEL2ID[f"B-{entity_label}"])
            else:  # "I"
                ner_tags.append(LABEL2ID[f"I-{entity_label}"])


    has_entity = any(t != LABEL2ID["O"] for t in ner_tags)
    if not has_entity:
        return None, None


    if len(tokens) < 5 or len(tokens) > 300:
        return None, None

    return tokens, ner_tags


#  test on one clause
print("Testing clause_to_iob2 on sample clause...\n")

sample_text = "Apple Inc agrees to pay Microsoft Corporation $500,000 by January 15, 2025 under New York law."

tokens, tags = clause_to_iob2(sample_text, nlp)

if tokens:
    print(f"{'Token':22s} {'Label'}")
    print("-" * 40)
    for token, tag in zip(tokens, tags):
        label  = ID2LABEL[tag]
        marker = " ←" if label != "O" else ""
        print(f"{token:22s} {label}{marker}")
else:
    print("No entities found in sample")

In [ ]:
import os
BASE = '/content/drive/MyDrive/LegalDocAnalyzer'

required = [
    'data/processed/tokenized_dataset',
    'models/ner/model.safetensors',
    'src/label_config.json',
    'src/class_weights.json',
    'src/entity_map.json',
]

print("=== DRIVE VERIFICATION ===")
for path in required:
    full_path = f'{BASE}/{path}'
    exists    = os.path.exists(full_path)
    print(f"  {'✅' if exists else '❌'} {path}")

Auto label Ledgar clauses

In [ ]:
import random
random.seed(42)

# ── settings ──
MAX_CLAUSES    = 10000  # process 10k clauses from 60k


BATCH_SIZE     = 32     # process 32 clauses at once

TARGET_LABELED = 8000   # stop when we have 8000 labeled examples


# shuffle so we get diverse clause types
all_texts = [ledgar_train[i]['text'] for i in range(len(ledgar_train))]
random.shuffle(all_texts)
selected_texts = all_texts[:MAX_CLAUSES]

print(f"Processing {MAX_CLAUSES} clauses in batches of {BATCH_SIZE}")
print(f"Target: {TARGET_LABELED} labeled examples")


ledgar_tokens   = []
ledgar_ner_tags = []
skipped         = 0
entity_counts   = Counter()

# ── batch processing ──
for batch_start in range(0, len(selected_texts), BATCH_SIZE):
    batch_texts = selected_texts[batch_start:batch_start + BATCH_SIZE]

    # process entire batch at once — much faster than one by one
    docs = list(nlp.pipe(batch_texts, batch_size=BATCH_SIZE))

    for doc in docs:
        text   = doc.text
        tokens = []
        tags   = []

        for token in doc:
            if token.is_space:
                continue

            word     = token.text
            ent_type = token.ent_type_
            ent_iob  = token.ent_iob_

            tokens.append(word)

            if ent_iob == "O" or ent_type not in SPACY_TO_ENTITY:
                tags.append(LABEL2ID["O"])
            else:
                entity_label = SPACY_TO_ENTITY[ent_type]
                if ent_iob == "B":
                    tags.append(LABEL2ID[f"B-{entity_label}"])
                    entity_counts[entity_label] += 1
                else:
                    tags.append(LABEL2ID[f"I-{entity_label}"])

        # filter
        has_entity = any(t != LABEL2ID["O"] for t in tags)
        valid_len  = 5 <= len(tokens) <= 300

        if not has_entity or not valid_len:
            skipped += 1
            continue

        ledgar_tokens.append(tokens)
        ledgar_ner_tags.append(tags)

    # progress update every 500 clauses
    processed = batch_start + len(batch_texts)
    if processed % 500 == 0 or processed >= MAX_CLAUSES:
        print(f"  Processed: {processed:5d} | "
              f"Labeled: {len(ledgar_tokens):5d} | "
              f"Skipped: {skipped:5d}")

    # stop early if  hit target
    if len(ledgar_tokens) >= TARGET_LABELED:
        print(f"\n Target reached: {len(ledgar_tokens)} examples")
        break

print(f"\n=== LEDGAR AUTO-LABELING COMPLETE ===")
print(f"  Clauses processed : {processed}")
print(f"  Successfully labeled: {len(ledgar_tokens)}")
print(f"  Skipped (no entity) : {skipped}")
print(f"\n=== ENTITY DISTRIBUTION ===")
for entity, count in sorted(entity_counts.items()):
    bar = "=" * int(count / max(entity_counts.values()) * 30)
    print(f"  {entity:15s}: {count:5d}  {bar}")